### 1. Imports and cleaned dataframe

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import tifffile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [1]:
final_df = pd.read_csv("county_image_and_poverty_rate.csv", dtype={"county_id": str})

final_df = final_df[["image_path", "poverty_rate"]].copy()
final_df["poverty_rate"] = final_df["poverty_rate"].astype(float)

print(final_df.shape)
print(final_df.head())
print(final_df["poverty_rate"].describe())

NameError: name 'pd' is not defined

In [65]:
def set_seed(seed=5527):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(5527)

### 2. Train / validation / test split

In [66]:
# 70% Training set, 30% temporary set
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=5527
)
# temporary set is split equally - 15% into validation and test sets
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=5527
)

# Standardize target y using training set only
y_train_mean = train_df["poverty_rate"].mean()
y_train_std = train_df["poverty_rate"].std()

print("y_train_mean:", y_train_mean)
print("y_train_std: ", y_train_std)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

y_train_mean: 14.12527573529412
y_train_std:  5.405674746427598
Train: (2176, 2)
Val:   (466, 2)
Test:  (467, 2)


In [67]:
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

### 3. Load TIF image

In [71]:
### 3. Load image
from PIL import Image
import torch.nn.functional as F

def load_image(path, target_size=(64, 64)):
    path = str(path)
    suffix = Path(path).suffix.lower()

    # Read image depending on file type
    if suffix in [".tif", ".tiff"]:
        img = tifffile.imread(path)
        img = np.array(img, dtype=np.float32)
    elif suffix in [".png", ".jpg", ".jpeg", ".webp"]:
        img = Image.open(path)
        img = np.array(img, dtype=np.float32)
    else:
        raise ValueError(f"Unsupported file type: {suffix} for {path}")

    img = np.squeeze(img)

    # If multi-channel, reduce to one channel
    if img.ndim == 3:
        if img.shape[-1] <= 4:          # channel-last
            img = img.mean(axis=-1)
        elif img.shape[0] <= 4:         # channel-first
            img = img.mean(axis=0)
        else:
            img = img[0]

    if img.ndim != 2:
        raise ValueError(f"Expected 2D image, got shape {img.shape} for {path}")

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img = np.clip(img, a_min=0, a_max=None)

    # Helpful for skewed nighttime-light values
    img = np.log1p(img)

    # Convert to tensor for resizing/padding
    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

    _, _, h, w = img_tensor.shape
    target_h, target_w = target_size

    # Preserve aspect ratio
    scale = min(target_h / h, target_w / w)
    new_h = max(1, int(round(h * scale)))
    new_w = max(1, int(round(w * scale)))

    img_tensor = F.interpolate(
        img_tensor,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    # Pad to exactly target_size
    pad_h = target_h - new_h
    pad_w = target_w - new_w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img_tensor = F.pad(
        img_tensor,
        (pad_left, pad_right, pad_top, pad_bottom),
        mode="constant",
        value=0.0
    )

    img = img_tensor.squeeze(0).squeeze(0).numpy()
    return img.astype(np.float32)

### 4. Compute normalization statistics from training set only

In [73]:
def compute_mean_std(df):
    total_sum = 0.0
    total_sq_sum = 0.0
    total_pixels = 0

    for path in df["image_path"]:
        img = load_image(path)
        total_sum += img.sum()
        total_sq_sum += (img ** 2).sum()
        total_pixels += img.size

    mean = total_sum / total_pixels
    var = total_sq_sum / total_pixels - mean ** 2
    std = np.sqrt(max(var, 1e-8))

    return float(mean), float(std)

### 5. Dataset and DataLoaders


In [ ]:
class CountyPovertyDataset(Dataset):
    def __init__(self, df, img_mean, img_std, y_mean, y_std):
        self.df = df.reset_index(drop=True).copy()
        self.img_mean = img_mean
        self.img_std = img_std
        self.y_mean = y_mean
        self.y_std = y_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = load_image(row["image_path"])
        img = (img - self.img_mean) / (self.img_std + 1e-6)
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)

        y = float(row["poverty_rate"])
        y = (y - self.y_mean) / (self.y_std + 1e-6)
        y = torch.tensor(y, dtype=torch.float32)

        return img, y

sample_img = load_image(train_df.iloc[0]["image_path"])
print(sample_img.shape)


(64, 64)
['C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning\\Project\\county_overlay_png\\county_overlay_nighttine_light\\01_01001_Autauga_polygon_under_tif.png', 'C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning\\Project\\county_overlay_png\\county_overlay_nighttine_light\\01_01003_Baldwin_polygon_under_tif.png', 'C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning\\Project\\county_overlay_png\\county_overlay_nighttine_light\\01_01005_Barbour_polygon_under_tif.png', 'C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning\\Project\\county_overlay_png\\county_overlay_nighttine_light\\01_01007_Bibb_polygon_under_tif.png', 'C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning\\Project\\county_overlay_png\\county_overlay_nighttine_light\\01_01009_Blount_polygon_under_tif.png', 'C:\\Users\\zinsc\\OneDrive\\Desktop\\MS DataScience\\CSCI 5527 - Deep Learning

In [77]:
batch_size = 32

train_dataset = CountyPovertyDataset(train_df, train_mean, train_std, y_train_mean, y_train_std)
val_dataset = CountyPovertyDataset(val_df, train_mean, train_std, y_train_mean, y_train_std)
test_dataset = CountyPovertyDataset(test_df, train_mean, train_std, y_train_mean, y_train_std)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

images, targets = next(iter(train_loader))
print("Batch image shape:", images.shape)   # [batch, 1, 64, 64]
print("Batch target shape:", targets.shape) # [batch]

Batch image shape: torch.Size([32, 1, 64, 64])
Batch target shape: torch.Size([32])


In [79]:
sample_img = load_image(train_df.iloc[0]["image_path"])
print(sample_img.shape)  

(64, 64)


### 6. CNN Regressor and Evaluation Functions


In [80]:
class CNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 -> 32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 -> 16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        x = self.features(x)
        embedding = torch.flatten(x, start_dim=1)
        out = self.regressor(x).squeeze(1)

        if return_embedding:
            return out, embedding
        return out

In [81]:
def evaluate_regression(model, loader, criterion, device, y_mean, y_std):
    model.eval()

    all_preds_std = []
    all_targets_std = []
    running_loss = 0.0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)
            loss = criterion(preds, targets)

            running_loss += loss.item() * images.size(0)

            all_preds_std.extend(preds.cpu().numpy())
            all_targets_std.extend(targets.cpu().numpy())

    all_preds_std = np.array(all_preds_std)
    all_targets_std = np.array(all_targets_std)

    # Convert back to original poverty-rate scale
    all_preds = all_preds_std * (y_std + 1e-6) + y_mean
    all_targets = all_targets_std * (y_std + 1e-6) + y_mean

    mse = mean_squared_error(all_targets, all_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    avg_loss = running_loss / len(loader.dataset)

    return {
        "loss": avg_loss,   # loss on standardized scale
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

### 7. Training model

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNNRegressor().to(device)

criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

Using device: cpu


In [83]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device,
                y_mean, y_std, epochs=25, patience=10):
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_rmse": [],
        "val_mae": [],
        "val_r2": []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate_regression(model, val_loader, criterion, device, y_mean, y_std)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_mae"].append(val_metrics["mae"])
        history["val_r2"].append(val_metrics["r2"])

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f} | "
            f"Val MAE: {val_metrics['mae']:.4f} | "
            f"Val R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

In [ ]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    y_mean=y_train_mean,
    y_std=y_train_std,
    epochs=25,
    patience=10
)

Epoch 01 | Train Loss: 0.3935 | Val Loss: 0.3871 | Val RMSE: 5.3750 | Val MAE: 4.0458 | Val R2: 0.0368
Epoch 02 | Train Loss: 0.3794 | Val Loss: 0.3871 | Val RMSE: 5.3496 | Val MAE: 4.0598 | Val R2: 0.0459
Epoch 03 | Train Loss: 0.3746 | Val Loss: 0.3810 | Val RMSE: 5.3658 | Val MAE: 3.9943 | Val R2: 0.0401
Epoch 04 | Train Loss: 0.3701 | Val Loss: 0.3804 | Val RMSE: 5.3700 | Val MAE: 3.9752 | Val R2: 0.0386
Epoch 05 | Train Loss: 0.3700 | Val Loss: 0.3758 | Val RMSE: 5.3207 | Val MAE: 3.9548 | Val R2: 0.0561
Epoch 06 | Train Loss: 0.3675 | Val Loss: 0.3770 | Val RMSE: 5.2838 | Val MAE: 3.9892 | Val R2: 0.0692
Epoch 07 | Train Loss: 0.3664 | Val Loss: 0.3774 | Val RMSE: 5.3242 | Val MAE: 3.9779 | Val R2: 0.0549
Epoch 08 | Train Loss: 0.3647 | Val Loss: 0.3723 | Val RMSE: 5.3222 | Val MAE: 3.9161 | Val R2: 0.0556
Epoch 09 | Train Loss: 0.3644 | Val Loss: 0.3791 | Val RMSE: 5.4158 | Val MAE: 3.9489 | Val R2: 0.0221
Epoch 10 | Train Loss: 0.3617 | Val Loss: 0.3721 | Val RMSE: 5.2723 | Val

### 8. Testing model

In [85]:
test_metrics = evaluate_regression(
    model,
    test_loader,
    criterion,
    device,
    y_mean=y_train_mean,
    y_std=y_train_std
)

print("\nTest Results")
print(f"Test Loss: {test_metrics['loss']:.4f}")
print(f"Test MSE:  {test_metrics['mse']:.4f}")
print(f"Test RMSE: {test_metrics['rmse']:.4f}")
print(f"Test MAE:  {test_metrics['mae']:.4f}")
print(f"Test R2:   {test_metrics['r2']:.4f}")

torch.save(model.state_dict(), "cnn_poverty_regressor.pt")
print("Saved cnn_poverty_regressor.pt")


Test Results
Test Loss: 0.3792
Test MSE:  28.0163
Test RMSE: 5.2930
Test MAE:  4.0343
Test R2:   0.0529
Saved cnn_poverty_regressor.pt


In [86]:
torch.save(model.state_dict(), "cnn_poverty_regressor.pt")
print("Saved cnn_poverty_regressor.pt")

Saved cnn_poverty_regressor.pt


### 9. Comparison: predicted vs actual

In [87]:
def predict_loader(model, loader, device, y_mean, y_std):
    model.eval()
    preds_std = []
    targets_std = []

    with torch.no_grad():
        for images, y in loader:
            images = images.to(device)
            outputs = model(images)

            preds_std.extend(outputs.cpu().numpy())
            targets_std.extend(y.numpy())

    preds_std = np.array(preds_std)
    targets_std = np.array(targets_std)

    preds = preds_std * (y_std + 1e-6) + y_mean
    targets = targets_std * (y_std + 1e-6) + y_mean

    return preds, targets

test_preds, test_targets = predict_loader(
    model,
    test_loader,
    device,
    y_mean=y_train_mean,
    y_std=y_train_std
)

pred_df = pd.DataFrame({
    "actual_poverty_rate": test_targets,
    "predicted_poverty_rate": test_preds
})

print(pred_df.head())
pred_df.to_csv("test_predictions.csv", index=False)

   actual_poverty_rate  predicted_poverty_rate
0             9.400000               14.284393
1            14.800000               15.199398
2             4.299999               11.456985
3            16.200000               13.056611
4            15.800000               13.518330
